In [ ]:
import random
#Cards implementation
class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = value
    def __repr__(self):
        return f"{self.color} {self.value}"
def Deck():
    colors = ['Red', 'Yellow', 'Blue', 'Green']
    values = [str(i) for i in range(10)] + ['Skip']
    deck = [Card(c, v) for c in colors for v in values]
    random.shuffle(deck)
    return deck
def initialise():
    deck = Deck()
    states = {
        # 5 cards for each player
        'p1' : [deck.pop() for _ in range(5)],
        'p2' : [deck.pop() for _ in range(5)],
        'p3' : [deck.pop() for _ in range(5)],
        # initial card to play with
        'top_card' : deck.pop(),
        'deck' : deck
    }
    return states

def get_valid_moves(hand, top_card):
    valid_moves = []
    for card in hand:
        # checks if same color or number for the valid move
        if (card.color == top_card.color or card.value == top_card.value):
            valid_moves.append(card)
    return valid_moves

def apply_move(states, move, player_key):
    new_state = {
        # copying cards 
        'p1' : states['p1'][:],
        'p2' : states['p2'][:],
        'p3' : states['p3'][:], 
        'top_card' : states['top_card'],
        'deck' : states['deck'][:]
    }
    # if player doesnt have the card, he/she draws one from deck
    if move is None:
        if new_state['deck']:
            drawn = new_state['deck'].pop()
            new_state[player_key].append(drawn)
        return new_state, False
    # if player has the card and is playing it 
    new_state[player_key].remove(move)
    # updates the top_card to play against
    new_state['top_card'] = move
    # checks if the played card is a skip
    is_skip = (move.value == 'Skip')
    return new_state, is_skip

def evaluate(state, player_key, strategy):
    # Determine who the opponents are
    opponents = [p for p in ['p1', 'p2', 'p3'] if p != player_key]
    
    c_ai = len(state[player_key])  # Cards in AI's hand
    # Average cards held by opponents
    c_opp = (len(state[opponents[0]]) + len(state[opponents[1]])) / 2.0
    # Number of Skip cards in hand 
    s = sum(1 for card in state[player_key] if card.value == 'Skip')
    
    if strategy == "defensive":
        # Value Skips higher and worry more about opponents finishing 
        return 50 - (5 * c_ai) + (4 * c_opp) + (6 * s)
    else:
        # Prioritize getting rid of your own cards quickly
        return 50 - (10 * c_ai) + (2 * c_opp) + (2 * s)